#**Ajuste Fino de Hiperparâmetros**

Estudos e testes a respoeito do ajuste de hiperparametros de uma rede neural MLP. No exemplo, é usado o dataset no Fashion-MNIST

## Configurações Prévias (imports, defs, etc.)

In [1]:
# if "google.colab" in sys.modules:
#     %pip install -q -U keras_tuner~=1.4.6

!pip install keras-tuner~=1.4.6

import sys
import sklearn
import numpy as np
import tensorflow as tf
import keras_tuner as kt

from tensorflow import keras
from tensorflow.keras import layers
from packaging import version

assert sys.version_info >= (3, 7)
assert version.parse(sklearn.__version__) >= version.parse("1.0.1")
assert version.parse(tf.__version__) >= version.parse("2.8.0")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 1.3 MB/s eta 0:00:00


In [2]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

##Modelo Baseado em VGG lite

In [3]:
def build_model(hp):
    inputs = keras.Input(shape=(32, 32, 3))
    x = inputs

    for i in range(hp.Int("blocks", 2, 4)):
        filters = hp.Choice(f"filters_{i}", [32, 64])

        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)

        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)

        x = layers.MaxPooling2D()(x)
        if hp.Boolean(f"dropout_{i}"):
            x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)

    x = layers.Dense(
        hp.Choice("dense_units", [64, 128]),
        activation="relu"
    )(x)

    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(10, activation="softmax")(x)

    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(
            hp.Choice("lr", [1e-3, 5e-4, 1e-4])
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

##Carregamento e preparo dos dados (CIFAR10)

In [4]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Normalização (0–255 -> 0–1)
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0


# Labels: (N,1) -> (N,)
y_train = y_train.flatten()
y_test  = y_test.flatten()

# Informações úteis
print("x_train shape:", x_train.shape)  # (50000, 32, 32, 3)
print("x_test shape:",  x_test.shape)   # (10000, 32, 32, 3)
print("Numero de classes:", len(np.unique(y_train)))

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
x_train shape: (50000, 32, 32, 3)
x_test shape: (10000, 32, 32, 3)
Numero de classes: 10


In [5]:
# val_size = np.random.permutation(len(x_train))
val_size = 5000

x_val, x_train = x_train[:val_size], x_train[val_size:]
y_val, y_train = y_train[:val_size], y_train[val_size:]

print("x_train:", x_train.shape)  # (45000, 32, 32, 3)
print("x_val:", x_val.shape)      # (5000, 32, 32, 3)

x_train: (45000, 32, 32, 3)
x_val: (5000, 32, 32, 3)


##RandomSearch

In [6]:
num_models = 4

tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=num_models,  # número de modelos testados
    executions_per_trial=2, # repetir treino (robustez)
    directory="tuner_dir",
    project_name="cifar10_search"
)

In [ ]:
tuner.search(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=5,
    batch_size=64
)

Trial 1 Complete [00h 52m 55s]
val_accuracy: 0.6960000097751617

Best val_accuracy So Far: 0.6960000097751617
Total elapsed time: 00h 52m 55s

Search: Running Trial #2

Value             |Best Value So Far |Hyperparameter
4                 |3                 |blocks
64                |32                |filters_0
True              |True              |dropout_0
64                |64                |filters_1
True              |False             |dropout_1
64                |128               |dense_units
0.001             |0.001             |lr
64                |32                |filters_2
False             |False             |dropout_2

Epoch 1/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 655s 917ms/step - accuracy: 0.3630 - loss: 1.7229 - val_accuracy: 0.5094 - val_loss: 1.3178
Epoch 2/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 656s 881ms/step - accuracy: 0.5530 - loss: 1.2659 - val_accuracy: 0.5978 - val_loss: 1.1072
Epoch 3/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 605s 857ms/step - accuracy: 0.6344 - loss: 1.0572 - 

##Recuperar o melhor modelo

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]

best_hp = tuner.get_best_hyperparameters(1)[0]

print("Melhores hiperparâmetros:")
print(best_hp.values)

##Treinar melhor modelo mais a fundo

In [ ]:
history = best_model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=20,
    batch_size=64
)